# Chapitre 4 · Construire un moteur d'autograd (solutions des exercices)

Ce notebook contient **uniquement les réponses aux six exercices** du notebook
du chapitre : la classe `Value` étape par étape, le gradient check de Sètondji,
le neurone entraîné, et l'extension `exp`. Le code de la leçon, lui, vit dans
le notebook du chapitre et dans le livre.

Si tu n'as pas encore vraiment essayé les exercices, referme ceci : ce chapitre
est LE chapitre IA débranchée, et le pacte vaut aussi pour les corrigés.

## Mise en place (reprise de la leçon)

Le capteur du chapitre 3, seul outil de la leçon dont les validations ont
besoin : il sert de juge indépendant aux gradient checks.

In [1]:
import math


def pente(f, x, h=1e-5):
    """Le capteur du chapitre 3 : de combien f(x) bouge quand on pousse x d'un cheveu."""
    return (f(x + h) - f(x - h)) / (2 * h)


# Verification rapide : sur f(x) = x**2, la pente exacte en x = 3 vaut 6.
print(f"pente(x**2, en 3) = {pente(lambda x: x ** 2, 3.0):.6f}")

pente(x**2, en 3) = 6.000000


### Exercice 1 · `Value`, version 1 : la boîte qui se souvient — niveau ●

`data` (la valeur), `_prev` (les parents, en ensemble), `_op` (l'opération qui
l'a produite). Les méthodes spéciales `__add__` et `__mul__` doivent calculer
ET déclarer la naissance, et accepter un opérande nu (`w * 8.0`) grâce au
`isinstance`.

In [2]:
class Value:
    def __init__(self, data, _prev=(), _op=""):
        self.data = data          # la valeur, un float ordinaire
        self._prev = set(_prev)   # ses parents dans le graphe
        self._op = _op            # l'operation qui l'a produite

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        return Value(self.data + other.data, (self, other), "+")

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        return Value(self.data * other.data, (self, other), "*")

    def __repr__(self):
        return f"Value(data={self.data})"

In [3]:
# Validation : le calcul se fait ET s'enregistre.
a = Value(2.0); b = Value(-3.0); c = Value(10.0)
d = a * b        # -6.0, ne de a et b par "*"
e = d + c        #  4.0, ne de d et c par "+"
assert isinstance(e, Value), "e doit etre une Value, pas un float"
assert d.data == -6.0 and e.data == 4.0, "les data doivent valoir -6.0 et 4.0"
assert d._op == "*" and e._op == "+", "_op doit garder le nom de l'operation"
assert d._prev == {a, b}, "les parents de d sont a et b"
assert e._prev == {d, c}, "les parents de e sont d et c"
assert a._prev == set() and a._op == "", "une feuille n'a ni parents ni operation"
assert (a + 1.0).data == 3.0, "l'autre operande peut arriver nu (isinstance)"
print("Exercice 1 OK : chaque calcul laisse sa trace.")

Exercice 1 OK : chaque calcul laisse sa trace.


### Exercice 2 · Les dérivées locales : `_backward` de `+` et `*` — niveau ●●

Le constructeur reçoit deux nouveautés (fournies) : `grad`, qui démarre à 0.0,
et `_backward`, le mode d'emploi du retour. À toi de l'écrire pour chaque
opération : l'**addition** transmet le gradient **tel quel**, la
**multiplication** **échange les valeurs de ses opérandes**. Écris-les avec
`=`, comme la version naïve de la leçon : le piège du nœud partagé sera déminé
à l'exercice 3, c'est voulu.

In [4]:
class Value:
    def __init__(self, data, _prev=(), _op=""):
        self.data = data
        self.grad = 0.0                     # la pente, inconnue pour l'instant
        self._backward = lambda: None       # le mode d'emploi du retour
        self._prev = set(_prev)
        self._op = _op

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), "+")

        def _backward():
            self.grad = out.grad            # derivee locale 1 : transmis tel quel
            other.grad = out.grad
        out._backward = _backward

        return out

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), "*")

        def _backward():
            self.grad = other.data * out.grad    # l'echange : la valeur de l'autre
            other.grad = self.data * out.grad
        out._backward = _backward

        return out

    def __repr__(self):
        return f"Value(data={self.data})"

In [5]:
# Validation : le retour a la main, sept gradients exacts.
a = Value(2.0); b = Value(-3.0); c = Value(10.0)
d = a * b          # -6.0
e = d + c          #  4.0
f = Value(-2.0)
L = e * f          # -8.0, notre " loss " miniature

L.grad = 1.0       # amorce : L bouge de 1 quand L bouge de 1
L._backward()      # remplit e.grad et f.grad
e._backward()      # remplit d.grad et c.grad
d._backward()      # remplit a.grad et b.grad

assert L.grad == 1.0, "l'amorce : L.grad = 1.0"
assert e.grad == -2.0 and f.grad == 4.0, "L = e*f : l'echange des valeurs"
assert d.grad == -2.0 and c.grad == -2.0, "e = d+c : transmis tel quel"
assert a.grad == 6.0 and b.grad == -4.0, "d = a*b : l'echange, encore"

# Le juge independant : le capteur du chapitre 3, sur a.
pente_capteur = pente(lambda av: ((av * -3.0) + 10.0) * -2.0, 2.0)
assert abs(pente_capteur - a.grad) < 1e-3, "le capteur doit confirmer a.grad = 6"
print(f"Exercice 2 OK. Capteur sur a : {pente_capteur:.6f} (moteur : {a.grad})")

Exercice 2 OK. Capteur sur a : 6.000000 (moteur : 6.0)


### Exercice 3 · La classe finale : accumulation, `tanh`, `backward()` — niveau ●●●

Trois chantiers pour clore le moteur :

- **l'accumulation** : un nœud utilisé sur plusieurs chemins reçoit plusieurs
  contributions, qui doivent s'**additionner** : `+=` partout dans les
  `_backward` (voilà pourquoi `grad` démarre à 0.0) ;
- **`tanh`** : ta première opération « toute faite ». Si `t = tanh(x)`, la
  dérivée locale vaut `1 - t**2`, lisible sur la sortie elle-même ;
- **`backward()`** : le tri topologique (un nœud n'entre dans la liste
  qu'APRÈS toute sa généalogie), puis l'amorce à 1.0 et la liste parcourue à
  l'envers. Règle absolue : un nœud ne distribue son gradient qu'une fois
  qu'il a reçu TOUTES ses contributions.

In [6]:
import math


class Value:
    def __init__(self, data, _prev=(), _op=""):
        self.data = data
        self.grad = 0.0
        self._backward = lambda: None
        self._prev = set(_prev)
        self._op = _op

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), "+")

        def _backward():
            self.grad += out.grad           # += : les contributions s'ADDITIONNENT
            other.grad += out.grad
        out._backward = _backward

        return out

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), "*")

        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        out._backward = _backward

        return out

    def tanh(self):
        t = math.tanh(self.data)
        out = Value(t, (self,), "tanh")

        def _backward():
            self.grad += (1 - t ** 2) * out.grad    # derivee locale : 1 - tanh**2
        out._backward = _backward

        return out

    def backward(self):
        ordre = []
        vus = set()

        def visiter(v):
            if v not in vus:
                vus.add(v)
                for parent in v._prev:
                    visiter(parent)
                ordre.append(v)
        visiter(self)

        self.grad = 1.0
        for v in reversed(ordre):
            v._backward()

    def __repr__(self):
        return f"Value(data={self.data})"

In [7]:
# Validation en trois temps : accumulation, tanh, backward automatique.
# 1) Le piege du noeud partage est demine.
a = Value(3.0)
s = a + a
s.grad = 1.0
s._backward()
assert a.grad == 2.0, f"a.grad doit valoir 2.0 (1.0 + 1.0), obtenu {a.grad}"

# 2) tanh : la valeur, et la derivee locale 1 - tanh**2.
x = Value(0.5)
y = x.tanh()
assert abs(y.data - math.tanh(0.5)) < 1e-12, "tanh doit utiliser math.tanh"
assert y._prev == {x} and y._op == "tanh", "un seul parent, _op = 'tanh'"
y.grad = 1.0
y._backward()
exacte = 1 - math.tanh(0.5) ** 2
assert abs(x.grad - exacte) < 1e-9, f"derivee locale attendue {exacte}, obtenue {x.grad}"

# 3) Le neurone entier, derive en UNE ligne : cinq gradients exacts.
x1, x2 = Value(1.0), Value(0.0)
w1, w2 = Value(2.0), Value(1.0)
b = Value(-0.9013877113318902)
o = (x1 * w1 + x2 * w2 + b).tanh()
o.backward()
assert abs(o.data - 0.8) < 1e-4, "la sortie doit valoir 0.8"
assert abs(w1.grad - 0.36) < 1e-6, f"w1.grad attendu 0.36, obtenu {w1.grad}"
assert w2.grad == 0.0, "w2 multiplie x2 = 0 : aucune influence, gradient nul"
assert abs(b.grad - 0.36) < 1e-6, f"b.grad attendu 0.36, obtenu {b.grad}"
assert abs(x1.grad - 0.72) < 1e-6, f"x1.grad attendu 0.72, obtenu {x1.grad}"
assert abs(x2.grad - 0.36) < 1e-6, f"x2.grad attendu 0.36, obtenu {x2.grad}"
print("Exercice 3 OK : ton moteur est complet.")

Exercice 3 OK : ton moteur est complet.


### Exercice 4 · Le gradient check de Sètondji — niveau ●●

Le trajet du chapitre 3 : 8 km payés 1 250 FCFA, tarif candidat `w = 100`
FCFA/km, erreur au carré. Construis le graphe avec TES `Value` et lance le
backward : ton moteur et le capteur doivent tomber d'accord sur -7200. Note le
clin d'œil : `ecart * ecart` est le nœud partagé de l'exercice 3 ; sans ton
`+=`, le moteur répondrait -3600, moitié de la vérité.

In [8]:
w = Value(100.0)
pred = w * 8.0                    # le prix predit : 800 FCFA
ecart = pred + (-1250.0)          # l'ecart au prix reel : -450
err = ecart * ecart               # l'erreur au carre : 202500 (et le noeud partage !)

err.backward()
print(f"moteur  : w.grad = {w.grad}")


def erreur_trajet(w):
    return (w * 8.0 - 1250.0) ** 2


print(f"capteur : {pente(erreur_trajet, 100.0)}")

moteur  : w.grad = -7200.0
capteur : -7200.000001466832


In [9]:
# Validation par gradient check : deux machines independantes, un seul nombre.
assert err.data == 202500.0, "l'erreur au carre doit valoir 202500"
assert w.grad == -7200.0, f"le moteur doit repondre -7200.0 exactement, obtenu {w.grad}"
assert abs(w.grad - pente(erreur_trajet, 100.0)) < 1e-3, "moteur et capteur doivent coincider"
print("Exercice 4 OK : ton moteur est valide, comme dans les labos.")

Exercice 4 OK : ton moteur est valide, comme dans les labos.


### Exercice 5 · Un neurone entraîné avec TON moteur — niveau ●●

Le refrain du livre (prédire, mesurer l'erreur, corriger, recommencer), sans
une ligne de PyTorch. `perte()` est fournie : elle fait l'aller et construit un
graphe **neuf** à chaque étape. À toi le cœur de la boucle : remise à zéro des
gradients, backward, descente.

In [10]:
notes = [[0.9, 0.8], [0.3, 0.2], [0.8, 0.6], [0.2, 0.4]]   # (maths, francais), sur 1
cibles = [1.0, -1.0, 1.0, -1.0]                             # admis / recale

w1, w2, b = Value(0.1), Value(-0.2), Value(0.05)            # depart quelconque


def perte():
    total = Value(0.0)
    for (xm, xf), cible in zip(notes, cibles):
        o = (w1 * xm + w2 * xf + b).tanh()      # predire
        e = o + (-cible)
        total = total + e * e                   # mesurer l'erreur (au carre)
    return total


for etape in range(101):
    loss = perte()                          # l'aller construit le graphe
    if etape % 20 == 0:
        print(f"étape {etape:3d} | loss = {loss.data:.4f}")
    w1.grad = w2.grad = b.grad = 0.0        # remise a zero (le += s'accumule !)
    loss.backward()                         # le retour : TES gradients
    for p in (w1, w2, b):
        p.data -= 0.5 * p.grad              # corriger : la descente du chapitre 3

loss_finale = perte()
print(f"final     | loss = {loss_finale.data:.4f}")

étape   0 | loss = 4.0822
étape  20 | loss = 0.0196
étape  40 | loss = 0.0163
étape  60 | loss = 0.0139
étape  80 | loss = 0.0122
étape 100 | loss = 0.0108
final     | loss = 0.0108


In [11]:
# Validation de l'entrainement : la loss a fondu, les predictions sont justes.
assert loss_finale.data < 0.05, f"loss finale attendue < 0.05, obtenue {loss_finale.data:.4f}"
for (xm, xf), cible in zip(notes, cibles):
    o = (w1 * xm + w2 * xf + b).tanh()
    assert o.data * cible > 0, f"prediction du mauvais signe pour ({xm}, {xf})"
    print(f"notes ({xm}, {xf}) -> {o.data:+.3f}   (cible {cible:+.0f})")
print("Exercice 5 OK : ton moteur vient de faire apprendre une machine.")

notes (0.9, 0.8) -> +0.990   (cible +1)
notes (0.3, 0.2) -> -0.944   (cible -1)
notes (0.8, 0.6) -> +0.933   (cible +1)
notes (0.2, 0.4) -> -0.945   (cible -1)
Exercice 5 OK : ton moteur vient de faire apprendre une machine.


### Exercice 6 · `exp` rejoint le moteur — niveau ●●●

La leçon l'a promis : toute fonction dont tu connais la dérivée locale peut
rejoindre le moteur en huit lignes, sur le moule de `tanh`. Preuve par
l'**exponentielle**, dont le chapitre 5 aura besoin. Sa dérivée locale est
encore plus simple que celle de tanh : la dérivée de `exp(x)`, c'est `exp(x)`
elle-même, déjà calculée dans la sortie. Écris `exp(v)` en fonction (elle
prend une `Value`, elle en renvoie une), sans toucher à la classe.

In [12]:
def exp(v):
    e = math.exp(v.data)
    out = Value(e, (v,), "exp")

    def _backward():
        v.grad += e * out.grad          # la derivee locale d'exp, c'est exp
    out._backward = _backward

    return out


In [13]:
# Validation : valeur, graphe, et gradient check contre le capteur.
x = Value(1.3)
y = exp(x)
assert isinstance(y, Value), "exp doit renvoyer une Value"
assert abs(y.data - math.exp(1.3)) < 1e-12, "exp doit utiliser math.exp"
assert y._prev == {x} and y._op == "exp", "un seul parent, _op = 'exp'"
y.backward()
assert abs(x.grad - math.exp(1.3)) < 1e-9, "la derivee locale d'exp est exp elle-meme"
assert abs(x.grad - pente(math.exp, 1.3)) < 1e-3, "le capteur doit confirmer"
print(f"Exercice 6 OK : exp a rejoint ton moteur (x.grad = {x.grad:.6f}).")

Exercice 6 OK : exp a rejoint ton moteur (x.grad = 3.669297).
